In [1]:
from google.colab import drive
drive.mount('/content/drive')
# if you downloaded a zip to Drive:
!unzip -q "/content/drive/MyDrive/FS2K.zip" -d /content/data


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/data/FS2K/anno_test.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/data/FS2K/anno_train.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/data/FS2K/photo/photo1/image0001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/data/FS2K/photo/photo1/image0002.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [2]:

import json
ROOT = "/content/data/FS2K"
!ls {ROOT} {ROOT}/photo {ROOT}/sketch
train = json.load(open(f"{ROOT}/anno_train.json"))
print(len(train)); print(train[0])
!ls {ROOT}/photo/photo1 | head -3
!ls {ROOT}/sketch/sketch1 | head -3

/content/data/FS2K:
anno_test.json	anno_train.json  photo	README.pdf  sketch

/content/data/FS2K/photo:
photo1	photo2	photo3

/content/data/FS2K/sketch:
sketch1  sketch2  sketch3
1058
{'image_name': 'photo1/image0110', 'skin_color': [163, 139], 'lip_color': [156.97750511247443, 82.51124744376278, 79.0], 'eye_color': [118.65178571428571, 72.25892857142857, 69.59821428571429], 'hair': 0, 'hair_color': 2, 'gender': 0, 'earring': 1, 'smile': 1, 'frontal_face': 1, 'style': 0}
image0001.jpg
image0002.jpg
image0003.jpg
sketch0001.jpg
sketch0002.jpg
sketch0003.jpg


In [3]:
import json, random, time
from pathlib import Path
from PIL import Image
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split
!pip -q install optuna wandb onnx onnxruntime
import optuna, wandb

In [4]:
!pip install torchmetrics

In [17]:
import json, random, time
from pathlib import Path
from PIL import Image
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import optuna, wandb
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn

ROOT = Path("/content/data/FS2K"); S = 128; dev = "cuda"
def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

def find_img(folder, stem):
    for ext in (".jpg", ".JPG", ".jpeg", ".JPEG", ".png", ".PNG"):
        p = folder / f"{stem}{ext}"
        if p.exists(): return p
    raise FileNotFoundError(f"{folder}/{stem}.*")

def sketch_path(name):
    src, base = name.split("/")
    num = base.replace("image", "")
    for k in (src[-1], "1", "2", "3"):
        try: return find_img(ROOT/"sketch"/f"sketch{k}", f"sketch{num}")
        except FileNotFoundError: pass
    raise FileNotFoundError(name)

def load_pair(r):
    d, b = r["image_name"].split("/")
    ph = Image.open(find_img(ROOT/"photo"/d, b)).convert("RGB").resize((S, S), Image.BICUBIC)
    sk = Image.open(sketch_path(r["image_name"])).convert("RGB").resize((S, S), Image.BICUBIC)
    t = lambda im: torch.from_numpy(np.array(im)).permute(2, 0, 1).float() / 127.5 - 1
    return t(ph), t(sk), r["style"]

train_all = json.load(open(ROOT/"anno_train.json"))
test_recs = json.load(open(ROOT/"anno_test.json"))
styles = [r["style"] for r in train_all]
tr_idx, va_idx = train_test_split(range(len(train_all)), test_size=0.15, stratify=styles, random_state=42)
train_recs = [train_all[i] for i in tr_idx]; val_recs = [train_all[i] for i in va_idx]
print(len(train_recs), len(val_recs), len(test_recs), np.bincount([r["style"] for r in val_recs]))

class FS2K(Dataset):
    def __init__(self, recs, train=False):
        self.d = [load_pair(r) for r in recs]; self.train = train   # preload into RAM
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        ph, sk, s = self.d[i]
        if self.train and random.random() < 0.5:     # identical flip for both
            ph, sk = ph.flip(-1), sk.flip(-1)
        return ph, sk, torch.tensor(s)

ds_tr, ds_va = FS2K(train_recs, True), FS2K(val_recs)

899 159 1046 [54 52 53]


In [6]:
def down(i, o, bn=True):
    L = [nn.Conv2d(i, o, 4, 2, 1, bias=False)]
    if bn: L.append(nn.BatchNorm2d(o))
    return nn.Sequential(*L, nn.LeakyReLU(0.2, True))
def up(i, o, drop):
    L = [nn.ConvTranspose2d(i, o, 4, 2, 1, bias=False), nn.BatchNorm2d(o)]
    if drop > 0: L.append(nn.Dropout(drop))
    return nn.Sequential(*L, nn.ReLU(True))

class Generator(nn.Module):
    def __init__(self, c=64, emb=16, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        # style map concatenated to input; style vector also concatenated at bottleneck
        self.d1 = down(3 + emb, c, False)      # 64
        self.d2 = down(c, c*2)                 # 32
        self.d3 = down(c*2, c*4)               # 16
        self.d4 = down(c*4, c*8)               # 8
        self.d5 = down(c*8, c*8)               # 4
        self.d6 = down(c*8, c*8)               # 2
        self.u1 = up(c*8 + emb, c*8, drop)     # 4
        self.u2 = up(c*16, c*8, drop)          # 8
        self.u3 = up(c*16, c*4, 0)             # 16
        self.u4 = up(c*8, c*2, 0)              # 32
        self.u5 = up(c*4, c, 0)                # 64
        self.out = nn.Sequential(nn.ConvTranspose2d(c*2, 3, 4, 2, 1), nn.Tanh())
    def forward(self, x, s):
        e = self.emb(s)
        em = e[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        d1 = self.d1(torch.cat([x, em], 1)); d2 = self.d2(d1); d3 = self.d3(d2)
        d4 = self.d4(d3); d5 = self.d5(d4); d6 = self.d6(d5)
        eb = e[:, :, None, None].expand(-1, -1, d6.size(2), d6.size(3))
        u1 = self.u1(torch.cat([d6, eb], 1))
        u2 = self.u2(torch.cat([u1, d5], 1)); u3 = self.u3(torch.cat([u2, d4], 1))
        u4 = self.u4(torch.cat([u3, d3], 1)); u5 = self.u5(torch.cat([u4, d2], 1))
        return self.out(torch.cat([u5, d1], 1))

class PatchD(nn.Module):
    def __init__(self, c=64, emb=16):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        self.net = nn.Sequential(
            down(6 + emb, c, False), down(c, c*2), down(c*2, c*4),
            nn.Conv2d(c*4, c*8, 4, 1, 1, bias=False), nn.BatchNorm2d(c*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(c*8, 1, 4, 1, 1))                       # logits grid
    def forward(self, x, y, s):
        em = self.emb(s)[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        return self.net(torch.cat([x, y, em], 1))

def winit(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)): nn.init.normal_(m.weight, 0, 0.02)

In [7]:
import torchvision.utils as vu
bce = nn.BCEWithLogitsLoss()
fixed = next(iter(DataLoader(ds_va, batch_size=9, shuffle=False)))   # fixed val photos for progress grids

@torch.no_grad()
def evaluate(G, dl):
    G.eval(); l1s, ss, n = 0, 0, 0
    for x, y, s in dl:
        x, y, s = x.to(dev), y.to(dev), s.to(dev)
        f = G(x, s); b = x.size(0)
        l1s += F.l1_loss(f, y, reduction="sum").item() / (3*S*S); n += b
        ss += ssim_fn((f+1)/2, (y+1)/2, data_range=1.0, reduction="sum").item()
    G.train(); return l1s/n, ss/n

def train(cfg, epochs, log=False, trial=None, log_every=10, save=None):
    seed_all(42)
    G = Generator(cfg["c"], cfg["emb"], cfg["drop"]).to(dev); D = PatchD(cfg["c"], cfg["emb"]).to(dev)
    G.apply(winit); D.apply(winit)
    oG = torch.optim.Adam(G.parameters(), cfg["lr_g"], betas=(0.5, 0.999))
    oD = torch.optim.Adam(D.parameters(), cfg["lr_d"], betas=(0.5, 0.999))
    dl = DataLoader(ds_tr, cfg["bs"], shuffle=True, drop_last=True, num_workers=2)
    dlv = DataLoader(ds_va, 32); best = -1
    for ep in range(1, epochs+1):
        agg = np.zeros(4); nb = 0
        for x, y, s in dl:
            x, y, s = x.to(dev), y.to(dev), s.to(dev)
            fake = G(x, s)
            # --- D step
            pr = D(x, y, s); pf = D(x, fake.detach(), s)
            d_real = bce(pr, torch.ones_like(pr)); d_fake = bce(pf, torch.zeros_like(pf))
            oD.zero_grad(); (0.5*(d_real + d_fake)).backward(); oD.step()
            # --- G step
            pf = D(x, fake, s)
            g_adv = bce(pf, torch.ones_like(pf)); g_l1 = F.l1_loss(fake, y)
            oG.zero_grad(); (g_adv + cfg["lam"]*g_l1).backward(); oG.step()
            agg += [d_real.item(), d_fake.item(), g_adv.item(), g_l1.item()]; nb += 1
        agg /= nb; vl1, vss = evaluate(G, dlv)
        if log:
            wandb.log({"D_real": agg[0], "D_fake": agg[1], "G_adv": agg[2], "G_L1": agg[3],
                       "val_L1": vl1, "val_SSIM": vss}, step=ep)
            if ep % log_every == 0 or ep == 1:
                with torch.no_grad():
                    G.eval(); f = G(fixed[0].to(dev), fixed[2].to(dev)); G.train()
                grid = vu.make_grid(torch.cat([fixed[0].to(dev), fixed[1].to(dev), f]), nrow=9, normalize=True, value_range=(-1, 1))
                wandb.log({"samples (photo/GT/gen)": wandb.Image(grid)}, step=ep)
        score = vss - vl1          # objective: high SSIM, low L1
        if save and score > best: best = score; torch.save(G.state_dict(), save)
        if trial is not None:
            trial.report(score, ep)
            if trial.should_prune(): raise optuna.TrialPruned()
    return G, vl1, vss

In [10]:
def objective(trial):
    cfg = dict(lr_g=trial.suggest_float("lr_g", 5e-5, 1e-3, log=True),
               lr_d=trial.suggest_float("lr_d", 5e-5, 1e-3, log=True),
               bs=trial.suggest_categorical("bs", [8, 16, 32]),
               c=trial.suggest_categorical("c", [32, 48, 64]),
               drop=trial.suggest_float("drop", 0.0, 0.5),
               emb=trial.suggest_categorical("emb", [8, 16, 32, 64]),
               lam=trial.suggest_float("lam", 10, 200, log=True))
    wandb.init(project="genai-ass1-task4", name=f"optuna-t{trial.number}", config=cfg, group="optuna", reinit=True)
    try:
        _, vl1, vss = train(cfg, epochs=15, log=True, trial=trial)
    finally:
        wandb.finish()
    return vss - vl1

study = optuna.create_study(direction="maximize", study_name="task4_cgan",
        storage="sqlite:///task4.db", load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=5))
study.optimize(objective, n_trials=15)
print(study.best_trial.number, study.best_value, study.best_params)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taha05 (taha05-fast) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
[I 2026-10-04 07:51:08,484] A new study created in RDB with name: task4_cgan
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


D_fake,█▅▆▃▂▁▂▂▂▂▃▃▃▃▃
D_real,█▄▅▂▂▁▂▁▂▁▂▃▂▃▃
G_L1,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁
G_adv,▁▄▃▅▇█▇▇▇█▇▆▆▆▅
val_L1,▆█▃▂▁▂▁▃▂▂▂▁▃▁▂
val_SSIM,▁▃▇▇█▇█▆▇▆▆▇▅▇▇
D_fake,0.33668
D_real,0.3623
G_L1,0.19535
G_adv,1.78292
val_L1,0.21162


[I 2026-10-04 07:54:18,705] Trial 0 finished with value: 0.2687161023761741 and parameters: {'lr_g': 0.00013924250314515922, 'lr_d': 0.00012061102266621194, 'bs': 16, 'c': 64, 'drop': 0.27891242833351576, 'emb': 32, 'lam': 142.66645644746592}. Best is trial 0 with value: 0.2687161023761741.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


...

[I 2026-10-04 08:19:36,939] Trial 13 pruned. 


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


D_fake,█▆▆▂▁▁▂▂▂▃▃▂▂▂▂
D_real,█▆▅▂▁▁▂▁▂▂▃▂▃▃▂
G_L1,█▃▃▂▂▂▂▂▂▁▁▁▁▁▁
G_adv,▁▂▂▅▇▇▇▇▇█▇▇▇▇▆
val_L1,▆█▂▃▁▂▁▃▂▂▁▁▃▁▁
val_SSIM,▁▃▇▇█▇▇▆▇▆▇▇▄▇▇
D_fake,0.31225
D_real,0.35538
G_L1,0.19123
G_adv,2.07414
val_L1,0.20662


[I 2026-10-04 08:22:53,649] Trial 14 finished with value: 0.27935875786675346 and parameters: {'lr_g': 0.00018054370655584302, 'lr_d': 0.00025262750602261834, 'bs': 16, 'c': 64, 'drop': 0.1894126121683004, 'emb': 32, 'lam': 168.66844575855475}. Best is trial 14 with value: 0.27935875786675346.


14 0.27935875786675346 {'lr_g': 0.00018054370655584302, 'lr_d': 0.00025262750602261834, 'bs': 16, 'c': 64, 'drop': 0.1894126121683004, 'emb': 32, 'lam': 168.66844575855475}


In [8]:
import optuna
study = optuna.load_study(study_name="task4_cgan", storage="sqlite:///task4.db")
print(len(study.trials), "trials")
print(study.best_trial.number, study.best_value, study.best_params)
study.trials_dataframe().to_csv("task4_optuna.csv", index=False)

15 trials
14 0.27935875786675346 {'lr_g': 0.00018054370655584302, 'lr_d': 0.00025262750602261834, 'bs': 16, 'c': 64, 'drop': 0.1894126121683004, 'emb': 32, 'lam': 168.66844575855475}


In [11]:
nbest = study.best_params
cfg = dict(lr_g=best["lr_g"], lr_d=best["lr_d"], bs=best["bs"], c=best["c"],
           drop=best["drop"], emb=best["emb"], lam=best["lam"])
wandb.init(project="genai-ass1-task4", name="final", config=cfg, group="final", reinit=True)
G, vl1, vss = train(cfg, epochs=100, log=True, log_every=10, save="/content/G_best.pt")
wandb.save("/content/G_best.pt"); wandb.finish()
print(vl1, vss)

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


D_fake,▃▂▂▁▁▁▂▂▂▁▂▂▂▆▂▁▂▂▂▂█▂▂▂▂▃▂▂▂▂▂▄▂▂▃▄▂▃▂▃
D_real,▂▁▁▁▁▂▂▅▂▂▂▅▂▂▂▂▂▂▂▂▂▂▂▂█▃▃▂▃▂▂▃▃▃▃▅▃▂▃▃
G_L1,██▇▇▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁
G_adv,▁▂▇▇▇▇█▆▆▅▆▅█▅▅▄▄▅▅▄▇▄▄▄▂▂▃▃▃▃▂▂▂▃▂▂▃▁▁▁
val_L1,▇█▅▃▃▃▄▂▃▂▃▂▃▄▂▃▂▃▃▃▃▃▁▂▂▃▃▃▃▃▁▁▂▃▃▃▃▂▂▂
val_SSIM,▁▄▆▆▅▃▅▆▅▇▃▆▄▆▄▄█▅▄▅█▅▅▄▃▄▄▅▆▅▄▅▅▅▄▅▄▄▅▅
D_fake,0.45136
D_real,0.49001
G_L1,0.10957
G_adv,1.47086
val_L1,0.20231


0.2023108020518561 0.48619691980709817


In [11]:
study = optuna.load_study(study_name="task4_cgan", storage="sqlite:///task4.db")
best = study.best_params
cfg = dict(lr_g=best["lr_g"], lr_d=best["lr_d"], bs=best["bs"], c=best["c"],
           drop=best["drop"], emb=best["emb"], lam=best["lam"])

G = Generator(cfg["c"], cfg["emb"], cfg["drop"]).to(dev)
G.load_state_dict(torch.load("/content/G_best.pt", map_location=dev))
G.eval()

Generator(
  (emb): Embedding(3, 32)
  (d1): Sequential(
    (0): Conv2d(35, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (d2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (d3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (d4): Sequential(
    (0): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (d5): Sequential(
    (0): Con

In [24]:
G = Generator(cfg["c"], cfg["emb"], cfg["drop"]).to(dev)
G.load_state_dict(torch.load("/content/G_best.pt", map_location=dev)); G.eval()
x = torch.randn(1, 3, S, S, device=dev); s = torch.tensor([1], device=dev)

import os
for f in ("/content/face2sketch.onnx", "/content/face2sketch.onnx.data"):
    if os.path.exists(f): os.remove(f)          # clear the old broken export

torch.onnx.export(G, (x, s), "/content/face2sketch.onnx",
    input_names=["photo", "style"], output_names=["sketch"],
    dynamic_axes={"photo": {0: "b"}, "style": {0: "b"}, "sketch": {0: "b"}},
    opset_version=17, dynamo=False)

print(round(os.path.getsize("/content/face2sketch.onnx") / 1e6, 1), "MB")   # expect ~100 MB, not 0.01 MB
print(os.path.exists("/content/face2sketch.onnx.data"))                      # expect False

/tmp/ipykernel_25456/3052748795.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(G, (x, s), "/content/face2sketch.onnx",


118.2 MB
False


In [18]:
dl_te = DataLoader(FS2K(test_recs), 32)
print("test L1, SSIM:", evaluate(G, dl_te))

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


test L1, SSIM: (0.2213269030690726, 0.49141784288687185)


In [13]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.4 MB/s eta 0:00:00


In [19]:
# per-style test metrics
for k in range(3):
    recs = [r for r in test_recs if r["style"] == k]
    print(k, len(recs), evaluate(G, DataLoader(FS2K(recs), 32)))

0 619 (0.16267084677372176, 0.542404309613254)
1 381 (0.32656133289620837, 0.3930392052557838)
2 46 (0.1390177823495174, 0.620148866072945)


In [22]:
!mkdir /content/drive/MyDrive/genai1task4

In [23]:
!cp /content/G_best.pt /content/face2sketch.onnx /content/task4.db /content/task4_optuna.csv /content/drive/MyDrive/genai1task4